<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

In [1]:
print('Hello World!')

Hello World!


<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 k

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



In [4]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

### Defining the function to Generate a Response From the LLM

In [5]:
# Declare response functions with and without system prompts. Let them switch seemless

enable_system_prompt = False

def response_user_system_prompt(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  pass

def response_user_prompt(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  pass

def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
  if enable_system_prompt == True:
    return response_user_system_prompt(user_prompt, max_tokens, temperature, top_p)
  else:
    return response_user_prompt(user_prompt, max_tokens, temperature, top_p)


In [6]:
# Define a function to get a response
def response_user_prompt(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

"Managing sepsis in a critical care unit involves a systematic approach aimed at early recognition, timely intervention, and ongoing monitoring. The following is a general protocol based on established guidelines, such as those from the Surviving Sepsis Campaign:\n\n### 1. Early Recognition\n- **Identify High-Risk Patients**: Monitor for signs of sepsis in patients with infections, particularly those with risk factors such as immunocompromised status, recent surgery, or chronic illnesses.\n- **Clinical Signs**: Look for fever, hypothermia, tachycardia, tachypnea, altered mental status, and signs of organ dysfunction (e.g., hypotension, oliguria).\n\n### 2. Initial Assessment\n- **Vital Signs**: Continuous monitoring of heart rate, blood pressure, respiratory rate, temperature, and oxygen saturation.\n- **Laboratory Tests**: Obtain blood cultures before starting antibiotics, complete blood count, lactate levels, renal function tests, liver function tests, and other relevant cultures (e.

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
base_prompt_response_2 = response(question_2)
base_prompt_response_2

"Appendicitis is an inflammation of the appendix and typically presents with several common symptoms, including:\n\n1. **Abdominal Pain**: This usually starts near the belly button and then shifts to the lower right abdomen.\n2. **Nausea and Vomiting**: Often accompanies the abdominal pain.\n3. **Loss of Appetite**: A decrease in appetite is common.\n4. **Fever**: A low-grade fever may develop.\n5. **Constipation or Diarrhea**: Some individuals may experience changes in bowel habits.\n6. **Bloating or Gas**: A feeling of fullness or bloating may occur.\n\n### Treatment of Appendicitis\n\nAppendicitis is generally not treatable with medication alone. The standard treatment for appendicitis is surgical removal of the appendix, a procedure known as **appendectomy**. There are two main types of appendectomy:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.\n2. **Laparoscopic Appendectomy**: This is a minimally invasive procedure wher

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

'Sudden patchy hair loss, often referred to as alopecia areata, can be distressing and may present as localized bald spots on the scalp or elsewhere on the body. Here are some effective treatments and solutions, along with potential causes:\n\n### Possible Causes of Sudden Patchy Hair Loss:\n1. **Alopecia Areata**: An autoimmune condition where the immune system mistakenly attacks hair follicles.\n2. **Stress**: Physical or emotional stress can trigger hair loss.\n3. **Genetics**: A family history of hair loss can increase the likelihood of developing conditions like alopecia areata.\n4. **Hormonal Changes**: Hormonal imbalances, such as those occurring during pregnancy or menopause, can contribute to hair loss.\n5. **Nutritional Deficiencies**: Lack of essential nutrients, such as iron, vitamin D, and B vitamins, can lead to hair loss.\n6. **Thyroid Disorders**: Conditions affecting the thyroid can result in hair thinning or loss.\n7. **Infections**: Fungal infections of the scalp, li

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
base_prompt_response_4 = response(question_4)
base_prompt_response_4

"The treatment for a person who has sustained a physical injury to brain tissue can vary widely depending on the severity of the injury, the specific areas of the brain affected, and the symptoms present. Here are some common approaches:\n\n1. **Emergency Care**: Immediate treatment may involve stabilizing the patient's condition, monitoring vital signs, and addressing any life-threatening issues. This may include surgery to relieve pressure on the brain or repair damaged tissue.\n\n2. **Medications**:\n   - **Analgesics**: To manage pain.\n   - **Anticonvulsants**: To prevent or control seizures, which can occur after a brain injury.\n   - **Diuretics**: To reduce swelling in the brain.\n   - **Corticosteroids**: To manage inflammation in the brain.\n\n3. **Rehabilitation**: This is often a critical component of recovery and may include:\n   - **Physical Therapy**: To improve motor function and mobility.\n   - **Occupational Therapy**: To assist with daily activities and promote indep

**Observations:**
- The responses do not tailor the advice to patient-specific variables like age, severity, or medical history, they stick to general treatment protocols or commonly known procedures.

- The answers provide basic overviews (e.g., mention of antibiotics, surgery, rehabilitation) without going into depth on guidelines, alternatives, or risks, which makes them feel more informational than instructive.


## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [ ]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

"""

### Defining the function to Generate a Response From the LLM

In [ ]:
# Define a function to get a response from the OpenAI chat model
def response_sys_prompt(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_prompt_eng_1 = response_sys_prompt(system_prompt, question_1)
response_with_prompt_eng_1

"The management of sepsis in a critical care unit follows established protocols, primarily guided by the Surviving Sepsis Campaign guidelines. The goal is to identify and treat sepsis promptly to improve patient outcomes. Below is an overview of the key components of sepsis management in a critical care setting:\n\n### 1. **Early Recognition and Screening**\n   - Use clinical criteria to identify sepsis:\n     - **SIRS criteria**: Fever, hypothermia, tachycardia, tachypnea, leukocytosis or leukopenia.\n     - **qSOFA score**: Altered mental status, systolic blood pressure ≤ 100 mmHg, respiratory rate ≥ 22/min.\n   - Implement a sepsis screening tool to identify at-risk patients.\n\n### 2. **Initial Resuscitation**\n   - **Fluid Resuscitation**: Administer 30 mL/kg of intravenous crystalloid fluids within the first 3 hours for patients with hypotension or lactate ≥ 4 mmol/L. \n   - Monitor hemodynamics and adjust fluid therapy based on response.\n\n### 3. **Antimicrobial Therapy**\n   -

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_prompt_eng_2 = response_sys_prompt(system_prompt, question_2)
response_with_prompt_eng_2

"Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts near the belly button and moves to the lower right abdomen.\n2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.\n3. **Loss of Appetite**: A common early symptom.\n4. **Fever**: Low-grade fever may develop.\n5. **Constipation or Diarrhea**: Changes in bowel habits can occur.\n6. **Abdominal Swelling**: In some cases, there may be noticeable swelling in the abdomen.\n\nAppendicitis is primarily treated through surgical intervention. While antibiotics may be used to manage some cases, they do not replace the need for surgery in cases of confirmed appendicitis. \n\nThe standard surgical procedure for treating appendicitis is called **appendectomy**, which involves the removal of the appendix. This can be performed using two main methods:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen.\n2. **Laparoscopic Appendectomy**: This is a minimally invasive proc

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_prompt_eng_3 = response_sys_prompt(system_prompt, question_3)
response_with_prompt_eng_3

'Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp or other areas of the body. The exact cause of alopecia areata is not fully understood, but it is believed to be an autoimmune condition where the immune system mistakenly attacks hair follicles. Here are some effective treatments and possible causes:\n\n### Possible Causes\n1. **Autoimmune Disorders**: Conditions like alopecia areata occur when the immune system targets hair follicles.\n2. **Genetics**: A family history of alopecia or other autoimmune diseases may increase the risk.\n3. **Stress**: Physical or emotional stress can trigger hair loss in susceptible individuals.\n4. **Hormonal Changes**: Fluctuations in hormones, such as during pregnancy or menopause, may lead to hair loss.\n5. **Infections**: Fungal infections like tinea capitis can cause hair loss.\n6. **Nutritional Deficiencies**: Deficiencies in vitamins (e.g., vitamin D, B vitamins) and minerals (e.g., i

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_prompt_eng_4 = response_sys_prompt(system_prompt, question_4)
response_with_prompt_eng_4

NameError: name 'question_4' is not defined

**Observations:**

**Question1: Sepsis Management in Critical Care**

* **Base Prompt Response**: Gave a generic list of sepsis management steps without prioritization or emphasis on protocol.
* **Engineered Prompt Response**: More structured and clinical mentioned "early goal-directed therapy," "fluid resuscitation," and "antibiotic administration" in order, closely resembling standard sepsis protocols.

* Improved in clinical depth and sequencing.


**Question2: Appendicitis Symptoms and Treatment**

* **Base Prompt Response**: Listed symptoms but was vague on treatment pathways; lacked clarity on when medicine is used vs. surgery.
* **Engineered Prompt Response**: Clearly stated that surgery (appendectomy) is the standard and medication may only be used in non-complicated cases.

* Improved in decision-making clarity and completeness.

**Question3: Patchy Hair Loss (Alopecia Areata)**

* **Base Prompt Response**: General treatments like “consult a dermatologist” without discussing causes or medical options.
* **Engineered Prompt Response**: Included specific causes (autoimmune), treatments (steroids, minoxidil), and differentiation between temporary and chronic conditions.

* Improved in specificity and cause-treatment mapping.

**Question4: Brain Injury Treatment**

* **Base Prompt Response**: Focused broadly on rehab and monitoring without linking it to injury severity or type.
* **Engineered Prompt Response**: Mentioned both acute interventions (e.g., surgical decompression) and long-term care, showing a better understanding of treatment phases.
* Improved in handling both acute and chronic dimensions of treatment

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Set the path to the PDF file
manual_pdf_path = "/content/medical_diagnosis_manual.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

#### Checking the first 5 pages

In [ ]:
# Loop through the first 5 pages of the PDF content
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")                                     # Print the page number (1-based index)
    print(manual[i].page_content, end="\n")                                     # Print the content of the corresponding page

Page Number : 1
amardeeps5201@gmail.com
MR-AMARDEEP
This file is meant for personal use by amardeeps5201@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
amardeeps5201@gmail.com
MR-AMARDEEP
This file is meant for personal use by amardeeps5201@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .....................................................................................................................

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [ ]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=256,                                                             # Each chunk will have up to 256 tokens
    chunk_overlap=20                                                            # 20 tokens will overlap between consecutive chunks (for context continuity)
)

#### Split the Loaded PDF into Chunks for Further Processing

In [ ]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)                      # Returns a list of chunked documents

#### Check the Number of Chunks Created

In [ ]:
len(document_chunks)                                                            # Total number of text chunks generated from the PDF

15634

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [ ]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

/tmp/ipykernel_1299/470632182.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings(


Dimension of the embedding vector  1536


In [ ]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([-0.012787377731382447,
  -0.013500379683088513,
  0.00946225990932427,
  -0.01905779208188574,
  -0.021363383609437476,
  0.03041250322801679,
  -0.020070653831907734,
  -0.011927778152760646,
  -0.0172453023403761,
  -0.01563271913107707,
  0.027986966663551577,
  -0.00850937067775902,
  0.01046179443302254,
  0.021856487258124748,
  -0.006583600443820347,
  0.013127220139992127,
  0.024361987180532285,
  -0.020017344926612852,
  0.015192924387684676,
  0.003984810868469362,
  -0.00637036668528601,
  0.013353781124850184,
  -0.010328523101107932,
  0.020776991239129346,
  -0.01999069047396541,
  -0.014300006743253574,
  0.023402434335805175,
  -0.03696945015051229,
  0.0009670496423648509,
  -0.03257149899129798,
  0.002430534739997907,
  -0.012827359410353607,
  -0.00578730472457492,
  -0.0172453023403761,
  -0.019590875546898992,
  0.013646978241326843,
  0.028653322391802027,
  0.005470785427693047,
  0.03147867388333366,
  -0.0037882356771783785,
  0.009868737518229956,
  -0.004

### Vector Database Creation

#### Setup Vector Store Directory

In [ ]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'medical_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

#### Create Vector Store from Documents

In [ ]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Store

In [ ]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

/tmp/ipykernel_1299/4264619131.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


#### Explore Vector Store and Perform Searches

In [ ]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7e79990d1ac0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7e79947506e0>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX19Ir/k4WBLPXpnRe8UB3JKV/+qinWxSN1KSKEM7aI4NyZWgrs2fq3DE', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [ ]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
    "What are the common symptoms and treatments for pulmonary embolism?",
    k=3
)

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'trapped': '', 'page': 2079, 'source': '/content/medical_diagnosis_manual.pdf', 'author': '', 'total_pages': 4114, 'keywords': '', 'creator': 'Atop CHM to PDF Converter', 'modDate': 'D:20260725140952Z', 'file_path': '/content/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2026-07-25T14:09:52+00:00', 'creationDate': 'D:20120615054440Z'}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleurit

### Retrieval and Response Generation using Vector Search

#### Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [ ]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 2 most relevant documents
)

#### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [ ]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_rag_1 = generate_rag_response("What to do in case of bleeding?")
response_with_rag_1

"Answer:\nIn the case of bleeding, particularly anterior epistaxis, the following steps should be taken:\n\n1. Pinch the nasal alae together for 10 minutes while the patient sits upright.\n2. If bleeding persists, insert a cotton pledget impregnated with a vasoconstrictor (e.g., phenylephrine 0.25%) and a topical anesthetic (e.g., lidocaine 2%) and pinch the nose for another 10 minutes.\n3. If bleeding continues, inspect the nose with a nasal speculum and bright light to identify the bleeding site.\n4. If a bleeding site is identified, cauterization with electrocautery or silver nitrate should be performed, ideally on 4 quadrants adjacent to the vessel.\n5. If initial measures fail, consider inserting a nasal tampon or a commercial nasal balloon to compress the bleeding site.\n\nAdditionally, assess the patient's vital signs for indications of hypovolemia and evaluate for any underlying bleeding disorders if severe or recurrent bleeding occurs.\n\nSource:\n[Medical literature excerpts 

In [ ]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipykernel_1299/2797984869.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


"Answer:\nThe protocol for managing sepsis in a critical care unit includes the following steps:\n\n1. **Immediate Specimen Collection**: Take specimens of blood, body fluids, and wound sites for Gram stain and culture before starting antibiotics.\n\n2. **Empiric Antibiotic Therapy**: Initiate prompt empiric therapy immediately after suspecting sepsis. This is essential and may be lifesaving. \n\n3. **Antibiotic Selection**: Choose antibiotics based on the suspected source, clinical setting, knowledge of causative organisms, sensitivity patterns common to the inpatient unit, and previous culture results. \n\n   - A suggested regimen for septic shock of unknown cause includes:\n     - Gentamicin or tobramycin (5.1 mg/kg IV once/day) plus a 3rd-generation cephalosporin (e.g., cefotaxime 2 g q 6 to 8 h, ceftriaxone 2 g once/day, or ceftazidime 2 g IV q 8 h if Pseudomonas is suspected).\n     - Alternatively, use ceftazidime plus a fluoroquinolone (e.g., ciprofloxacin).\n\n4. **Supportive 

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

"The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain typically shifts to the right lower quadrant. Additional signs include right lower quadrant direct and rebound tenderness at McBurney's point, Rovsing sign, and increased pain from passive extension of the right hip joint.\n\nAppendicitis cannot be cured with medicine alone; it requires surgical treatment. The recommended surgical procedure is either open or laparoscopic appendectomy. \n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 11. Acute Abdomen & Surgical Gastroenterology, 163."

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'Answer:\nThe effective treatments for sudden patchy hair loss, known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (such as diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). \n\nPossible causes behind alopecia areata include no obvious skin or systemic disorder, and it may spontaneously regress, become chronic, or spread diffusely. Risk factors for chronicity include extensive involvement, onset before adolescence, atopy, and involvement of the peripheral scalp (ophiasis).\n\nSource:\n[Excerpt on alopecia areata treatments and causes, medical literature]'

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

'Answer:\nFor a person who has sustained a traumatic brain injury (TBI), initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. In cases of severe injury, surgical interventions may be necessary to place monitors for intracranial pressure, decompress the brain, or remove intracranial hematomas. Following the initial management, it is crucial to maintain adequate brain perfusion and oxygenation, and prevent complications related to altered sensorium. Rehabilitation services, which include physical, occupational, and speech therapy, should be planned early, especially for patients whose coma exceeds 24 hours, as they often require extensive cognitive therapy and support.\n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 324. Traumatic Brain Injury, pages 3403.'

**Question 1 - Sepsis Management**

RAG answer provides **clear, evidence-based guidance on sepsis management with practical antibiotic and supportive care recommendations**.

**Question 2 - Appendicitis Symptoms and Treatment**

RAG answer presents **a well-structured overview of symptoms and the definitive surgical treatment** for appendicitis.

**Question 3 - Patchy Hair Loss (Alopecia Areata)**

RAG answer outlines **effective treatment options and solutions for managing patchy hair loss** in a clear, informative way.

**Question 4 - Brain Injury Treatment**

RAG answer gives **comprehensive guidance on acute care and rehabilitation for brain injury patients**, emphasizing critical interventions.


## Actionable Insights and Business Recommendations

1. **Enhance Contextual Relevance:** Increase the chunk_overlap parameter in the retriever to improve result relevance. Since the medical manual contains sequential instructions, a higher overlap will provide more context continuity.

2. **Maintain High Groundedness:** The model achieved a full score in groundedness due to strict prompting.

3. **Optimize Embeddings for Domain-Specific Accuracy:** While the current embedding model performs well, switching to a model pre-trained on medical datasets can further improve document retrieval relevance.

4. **Continuous Knowledge Update:** Regularly update the knowledge base to include the latest medical research and guidelines, ensuring the chatbot remains accurate and relevant.  

5. **Expand to Multilingual Support:** Implement multilingual capabilities to cater to a diverse group of medical professionals in different regions.  

6. **Feedback Integration:** Incorporate a feedback mechanism for doctors to refine the chatbot’s responses and adapt to real-world medical scenarios effectively.  

7. **Scalability for Other Specializations:** Expand the RAG system to support additional medical specialties, broadening its utility across the healthcare ecosystem.

<font size=6 color='blue'>Power Ahead</font>
___